# 01 - Acquisition des données

Téléchargement des 7 datasets du défi **« Elections municipales 2026 et enjeux locaux »** :

| # | Dataset | Source |
|---|---------|--------|
| 1 | Communes SRU | MTE / data.gouv.fr |
| 2 | Carte des loyers 2025 (4 fichiers) | MTE / data.gouv.fr |
| 3 | Éducation prioritaire | MENJ / data.education.gouv.fr |
| 4 | Effectifs élèves par école | MENJ / data.education.gouv.fr |
| 5 | Personnels 1er degré | MENJ / data.education.gouv.fr |
| 6 | Revenus & Pauvreté (FiloSoFi) | INSEE |
| 7 | Géométrie départements | GitHub (france-geojson) |

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import DATA_RAW, DATA_EXTERNAL

## Téléchargement

In [6]:
%run ../src/download_data.py

[SKIP] donnees-sru-data-gouv-2025-v2.csv (already exists, 0.3 MB)
[SKIP] pred-app-mef-dhup.csv (already exists, 4.8 MB)
[SKIP] pred-app12-mef-dhup.csv (already exists, 4.8 MB)
[SKIP] pred-app3-mef-dhup.csv (already exists, 4.8 MB)
[SKIP] pred-mai-mef-dhup.csv (already exists, 4.8 MB)
[SKIP] fr-en-etablissements-ep.csv (already exists, 2.2 MB)
[SKIP] fr-en-ecoles-effectifs-nb_classes.csv (already exists, 158.1 MB)
[SKIP] fr-en-indicateurs_personnels_etablissements1d.csv (already exists, 8.2 MB)
[DOWNLOAD] filosofi_commut.csv
  [ERROR] 404 Client Error: Not Found for url: https://www.insee.fr/fr/statistiques/fichier/6457248/base-filosofi-communes-2021.zip
[SKIP] departements.geojson (already exists, 0.6 MB)

Done!


## Vérification

In [7]:
print(f"=== data/raw ({sum(f.stat().st_size for f in DATA_RAW.iterdir() if f.is_file()) / 1e6:.1f} MB) ===")
for f in sorted(DATA_RAW.iterdir()):
    if f.is_file():
        print(f"  {f.name:60s} {f.stat().st_size / 1e6:8.2f} MB")

print(f"\n=== data/external ===")
for f in sorted(DATA_EXTERNAL.iterdir()):
    if f.is_file():
        print(f"  {f.name:60s} {f.stat().st_size / 1e6:8.2f} MB")

=== data/raw (187.9 MB) ===
  donnees-sru-data-gouv-2025-v2.csv                                0.35 MB
  fr-en-ecoles-effectifs-nb_classes.csv                          158.06 MB
  fr-en-etablissements-ep.csv                                      2.23 MB
  fr-en-indicateurs_personnels_etablissements1d.csv                8.24 MB
  pred-app-mef-dhup.csv                                            4.77 MB
  pred-app12-mef-dhup.csv                                          4.75 MB
  pred-app3-mef-dhup.csv                                           4.76 MB
  pred-mai-mef-dhup.csv                                            4.76 MB

=== data/external ===
  departements.geojson                                             0.57 MB


In [8]:
import pandas as pd
from src.clean_data import load_csv

files_to_check = [
    ("SRU", DATA_RAW / "donnees-sru-data-gouv-2025-v2.csv", ";"),
    ("Loyers (appartements)", DATA_RAW / "pred-app-mef-dhup.csv", ";"),
    ("Éducation prioritaire", DATA_RAW / "fr-en-etablissements-ep.csv", ";"),
    ("Effectifs élèves", DATA_RAW / "fr-en-ecoles-effectifs-nb_classes.csv", ";"),
    ("Personnels 1er degré", DATA_RAW / "fr-en-indicateurs_personnels_etablissements1d.csv", ";"),
]

for label, path, sep in files_to_check:
    if not path.exists():
        print(f"{label}: FICHIER MANQUANT ({path.name})")
        continue
    df = load_csv(path, sep=sep, nrows=5)
    print(f"\n{label} ({path.name})")
    print(f"  Colonnes: {list(df.columns)}")
    print(f"  Shape: {df.shape}")


SRU (donnees-sru-data-gouv-2025-v2.csv)
  Colonnes: ['hexagone_drom', 'Region', 'Departement', 'Code_Departement', 'Code_INSEE_commune', 'Nom_commune', 'Population_municipale_01_01_2025', 'Code_SIREN_EPCI', 'Nom_EPCI', 'EPCI_SRU', 'Code_unité_urbaine', 'Nom_unité_urbaine', 'UU_SRU', 'Commune_isolée_article_L_302_5_CCH', 'Commune_sru_au_01_01_2025', 'Commune_sru_au_01_01_2024', 'Nombre_lls_ Inventaire_au_01_01_2024', 'Taux_SRU_au_01_01_2024', 'commune_deficitaire', 'Commune_carencée', 'Commune_exemptée_2023_2025', 'Taux_cible_commune_2023_2025', 'Prélèvement_net_2025_dont_majoration']
  Shape: (5, 23)

Loyers (appartements) (pred-app-mef-dhup.csv)
  Colonnes: ['id_zone', 'INSEE_C', 'LIBGEO', 'EPCI', 'DEP', 'REG', 'loypredm2', 'lwr.IPm2', 'upr.IPm2', 'TYPPRED', 'nbobs_com', 'nbobs_mail', 'R2_adj']
  Shape: (5, 13)

Éducation prioritaire (fr-en-etablissements-ep.csv)
  Colonnes: ['UAI', 'EP 2022-2023', 'Nom', 'Type', 'Statut', 'Académie', 'Département', 'Commune', 'Région', 'UAI tête de 

In [9]:
filosofi_zip = DATA_RAW / "base-filosofi-communes-2021.zip"
if filosofi_zip.exists():
    import zipfile
    with zipfile.ZipFile(filosofi_zip) as z:
        print("Fichiers dans l'archive FiloSoFi:")
        for name in z.namelist():
            print(f"  {name} ({z.getinfo(name).file_size / 1e6:.1f} MB)")
else:
    print(f"FiloSoFi: FICHIER MANQUANT ({filosofi_zip.name})")

FiloSoFi: FICHIER MANQUANT (base-filosofi-communes-2021.zip)
